# Final Project: Bank Customer Churn Analysis & Reporting
#### Course: Data Science Programming Languages (Lab)  
#### Dataset: Bank Customer Churn Records (Kaggle) 
-----------------
#### name of first student: Mustafa Mahmoud Hassouna
#### ID of first student: 120258131
--------------
#### name of second student: Hamid Ibrahim Miqdad
#### ID of second student: 120251119
---

In [58]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Setup Kaggle directory and credentials  هاد جزئية ال Kaggle

kaggle_path = Path.home() / ".kaggle"
kaggle_path.mkdir(exist_ok=True)

token_file = kaggle_path / "access_token"
token_file.write_text("******************************")

# Download and unzip dataset via Kaggle API     هان يا حامد تحميل مع فك ضغط
!kaggle datasets download -d radheshyamkollipara/bank-customer-churn --unzip

Dataset URL: https://www.kaggle.com/datasets/radheshyamkollipara/bank-customer-churn
License(s): other




  0%|          | 0.00/307k [00:00<?, ?B/s]
100%|##########| 307k/307k [00:00<00:00, 340kB/s]
100%|##########| 307k/307k [00:00<00:00, 340kB/s]


In [59]:
# Load CSV file into pandas DataFrame     وهان كمان تحميل الCSV
df = pd.read_csv("Customer-Churn-Records.csv")


print("Dataset Shape (Rows, Columns):", df.shape)

print("\nColumn Names:")

print(df.columns.tolist())

print("\nData Types:")

print(df.dtypes)

# Display   first 5   rows
df.head()

Dataset Shape (Rows, Columns): (10000, 18)

Column Names:
['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Complain', 'Satisfaction Score', 'Card Type', 'Point Earned']

Data Types:
RowNumber               int64
CustomerId              int64
Surname                   str
CreditScore             int64
Geography                 str
Gender                    str
Age                     int64
Tenure                  int64
Balance               float64
NumOfProducts           int64
HasCrCard               int64
IsActiveMember          int64
EstimatedSalary       float64
Exited                  int64
Complain                int64
Satisfaction Score      int64
Card Type                 str
Point Earned            int64
dtype: object


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


### Dataset Row Representation:
Each row in this dataset represents an individual bank customer's profile and financial interaction record. 
It contains 18 attributes including personal demographics (Geography, Gender, Age), account status (CreditScore, Balance, Tenure), engagement metrics (NumOfProducts, HasCrCard, IsActiveMember), behavioral responses (Complain, Satisfaction Score, Card Type, Point Earned), and the target variable (`Exited`), which indicates whether the customer has left the bank (1) or remained (0).

In [60]:
# Extract first 20 rows into pure Python data structure
raw_20 = df.head(20).to_dict(orient='records')


# 1. Lists & Dictionaries
credit_scores_list = [row['CreditScore'] for row in raw_20]

customer_lookup_dict = {row['CustomerId']: row['Surname'] for row in raw_20}


# 2_. List Comprehension
high_credit_customers = [row['Surname'] for row in raw_20 if row['CreditScore'] > 700]

# 3 -. map() with lambda

salaries_in_thousands = list(map(lambda row: round(row['EstimatedSalary'] / 1000, 2), raw_20))

# 4. filter() with lambda

active_customers = list(filter(lambda row: row['IsActiveMember'] == 1, raw_20))

# 5. Sorting

sorted_by_balance = sorted(raw_20, key=lambda row: row['Balance'], reverse=True)

# 6. Custom Function with Docstring

def calculate_loyalty_ratio(age: int, tenure: int) -> float:
    """
    Calculate customer loyalty ratio based on tenure relative to age.
    
    Parameters:  المدخلات 
        age (int): Customer's age in years.
        tenure (int): Number of years customer has stayed with the bank.
        
    Returns: الراجع 
        float: Loyalty percentage ratio.
    """
    if age <= 0:
        
        return 0.0
        
    return round((tenure / age) * 100, 2)

loyalty_scores = [calculate_loyalty_ratio(r['Age'], r['Tenure']) for r in raw_20]

# Display sample results from pure Python operations

print("High Credit Customers (>700):", high_credit_customers)

print("Salaries in Thousands (First 5):", salaries_in_thousands[:5])
print("Active Customers Count:", len(active_customers))
print("Loyalty Ratios (First 5):", loyalty_scores[:5])





High Credit Customers (>700): ['Mitchell', 'Bartlett', 'Hao']
Salaries in Thousands (First 5): [101.35, 112.54, 113.93, 93.83, 79.08]
Active Customers Count: 10
Loyalty Ratios (First 5): [4.76, 2.44, 19.05, 2.56, 4.65]


In [61]:
# Parent Class
class BankCustomer:
    
    def __init__(self, customer_id: int, surname: str, credit_score: int, balance: float, geography: str):
        self.customer_id = customer_id
        self.surname = surname
        
        self.credit_score = credit_score
        
        self._balance = balance  # Protected attribute for encapsulation هاد للحماي
        
        self.geography = geography

    # Encapsulation: Getter
    def get_balance(self) -> float:
        return self._balance
        

    # Encapsulation: Setter with validation
    
    def set_balance(self, amount: float):
        if amount < 0:
            raise ValueError("Balance cannot be negative.")
        self._balance = amount

    def calculate_risk(self) -> str:

        
        if self.credit_score < 580:
            return "High Risk"
            
        elif self.credit_score < 700:
            
            return "Medium Risk"
            
        else:
            return "Low Risk"

    def display_info(self) -> str:
        
        return f"Customer ID: {self.customer_id} | Name: {self.surname} | Country: {self.geography}"


# Child Class 1

class ActiveCustomer(BankCustomer):
    
    def __init__(self, customer_id: int, surname: str, credit_score: int, balance: float, geography: str, satisfaction_score: int):
        super().__init__(customer_id, surname, credit_score, balance, geography)
        
        self.satisfaction_score = satisfaction_score

    # Polymorphism  : MethodOverriding
    
    def calculate_risk(self) -> str:
        
        base_risk = super().calculate_risk()
        if self.satisfaction_score >= 4 and base_risk == "Medium Risk":
            
            return "Low Risk (Satisfied Member)"
        return base_risk

    def display_info(self) -> str:
        
        return f"[Active Member] {super().display_info()} | Satisfaction: {self.satisfaction_score}/5"


# Child Class 2
class CreditCardCustomer(BankCustomer):   # ما اطول هان #
    
    def __init__(self, customer_id: int, surname: str, credit_score: int, balance: float, geography: str, card_type: str, point_earned: int):
        super().__init__(customer_id, surname, credit_score, balance, geography)
        self.card_type = card_type
        self.point_earned = point_earned

    #        Polymorphism:Method Overriding
    def calculate_risk(self) -> str:
        
        base_risk = super().calculate_risk()
        if self.card_type.upper() == "DIAMOND" and self.point_earned > 400:
            return f"{base_risk} (VIP Loyalty Status)"
            
        return base_risk

    def display_info(self) -> str:

        
        return f"[Credit Card Member] {super().display_info()} | Card: {self.card_type} | Points: {self.point_earned}"


# Demonstration using actual dataset rows

customer_obj1 = ActiveCustomer(
    customer_id=df.loc[0, 'CustomerId'],
    surname=df.loc[0, 'Surname'],
    credit_score=df.loc[0, 'CreditScore'],
    balance=df.loc[0, 'Balance'],
    geography=df.loc[0, 'Geography'],
    satisfaction_score=df.loc[0, 'Satisfaction Score']
)
# حلو الترتيب هان :( 
customer_obj2 = CreditCardCustomer(
    customer_id=df.loc[1, 'CustomerId'],
    surname=df.loc[1, 'Surname'],
    credit_score=df.loc[1, 'CreditScore'],
    balance=df.loc[1, 'Balance'],
    geography=df.loc[1, 'Geography'],
    card_type=df.loc[1, 'Card Type'],
    point_earned=df.loc[1, 'Point Earned']
)




print(customer_obj1.display_info(), "| Assessed Risk:", customer_obj1.calculate_risk())
print(customer_obj2.display_info(), "| Assessed Risk:", customer_obj2.calculate_risk())

[Active Member] Customer ID: 15634602 | Name: Hargrave | Country: France | Satisfaction: 2/5 | Assessed Risk: Medium Risk
[Credit Card Member] Customer ID: 15647311 | Name: Hill | Country: Spain | Card: DIAMOND | Points: 456 | Assessed Risk: Medium Risk (VIP Loyalty Status)


In [62]:
# Custom Exception Class
class InvalidCreditScoreError(Exception):
    """Exception raised when a credit score falls outside the valid banking range (300 to 850)."""
    def __init__(self, score: int, message: str = "Credit score must be between 300 and 850"):
        
        self.score = score
        self.message = f"{message}. Received invalid value: {score}"
        super().__init__(self.message)


# Function demonstrating exception raising and validation logic   
def validate_customer_credit_score(record: dict):
    
    score = record.get("CreditScore", 0)
    if not (300 <= score <= 850):
        raise InvalidCreditScoreError(score)
    return True


# Demonstration: Catching the custom exception with practical test cases
test_records = [
    
    {"CustomerId": 1001, "CreditScore": 720},  # Valid record
    {"CustomerId": 1002, "CreditScore": 950},  # Invalid record (too high)
    {"CustomerId": 1003, "CreditScore": 250}   # Invalid record (too low)
]

for record in test_records:
    
    try:
        validate_customer_credit_score(record)
        print(f"Customer {record['CustomerId']}: Credit score {record['CreditScore']} is valid.")
    except InvalidCreditScoreError as e:
        
        print(f"Caught Exception for Customer {record['CustomerId']}: {e}")

Customer 1001: Credit score 720 is valid.
Caught Exception for Customer 1002: Credit score must be between 300 and 850. Received invalid value: 950
Caught Exception for Customer 1003: Credit score must be between 300 and 850. Received invalid value: 250


In [63]:
# Convert numerical columns to 1D NumPy arrays

credit_scores_arr = df['CreditScore'].values
balances_arr = df['Balance'].values

salaries_arr = df['EstimatedSalary'].values

# 1. Vectorized Calculation: Balance-to-Salary Ratio



#_------------------------------_____------------ Nice
# Handles potential division by zero safely



balance_salary_ratio = np.where(salaries_arr > 0, balances_arr / salaries_arr, 0.0)

# 2. Boolean Filtering: Extract customers with Credit Score > 750
high_credit_mask = credit_scores_arr > 750
high_credit_scores = credit_scores_arr[high_credit_mask]



# 3. Statistical Summary on NumPy arrays

num_stats = {
    'Mean': np.mean(credit_scores_arr),
    'Minimum': np.min(credit_scores_arr),
    
    'Maximum': np.max(credit_scores_arr),
    'Standard Deviation': np.std(credit_scores_arr)
}

print("=== NumPy Statistical Summary   (Credit Scores) ==")


for metric, val in num_stats.items():
    print(f"{metric}: {val:.2f}")
    

print("\nVectorized Balance-to-Salary Ratio (First 5):", np.round(balance_salary_ratio[:5], 4))
print("High Credit Score Customers Count (> 750):", len(high_credit_scores))

=== NumPy Statistical Summary   (Credit Scores) ==
Mean: 650.53
Minimum: 350.00
Maximum: 850.00
Standard Deviation: 96.65

Vectorized Balance-to-Salary Ratio (First 5): [0.     0.7447 1.4014 0.     1.5871]
High Credit Score Customers Count (> 750): 1598


In [64]:
# Create a copy for cleaning operations
df_clean = df.copy()

# 1. Standardize Column Names: Convert to snake_case (lowercase, remove spaces)
clean_column_names = {col: col.strip().lower().replace(" ", "_") for col in df_clean.columns}
df_clean.rename(columns=clean_column_names, inplace=True)

# 2. Handle Missing Values (Check & Fill)
missing_before = df_clean.isnull().sum().sum()
df_clean.fillna({'balance': 0.0}, inplace=True)

# 3. Detect and Remove Duplicate Rows
duplicates_count = df_clean.duplicated().sum()

df_clean.drop_duplicates(inplace=True)

# 4     <___. Data Type Conversions: Convert categorical strings to category type
categorical_columns = ['geography', 'gender', 'card_type']

for col in categorical_columns:
    df_clean[col] = df_clean[col].astype('category')
    

# 5   . Text Standardization: Trim whitespace and title case customer surnames
df_clean['surname'] = df_clean['surname'].astype(str).str.strip().str.title()

print("=== Data Cleaning Report ======")

print(f"Missing Values Cleaned: {missing_before}")
print(f"Duplicates Removed: {duplicates_count}")
print("\nUpdated Cleaned Columns:")
print(df_clean.columns.tolist())
print("\nCleaned Dataset Shape:", df_clean.shape)

=== Data Cleaning Report ======
Missing Values Cleaned: 0
Duplicates Removed: 0

Updated Cleaned Columns:
['rownumber', 'customerid', 'surname', 'creditscore', 'geography', 'gender', 'age', 'tenure', 'balance', 'numofproducts', 'hascrcard', 'isactivemember', 'estimatedsalary', 'exited', 'complain', 'satisfaction_score', 'card_type', 'point_earned']

Cleaned Dataset Shape: (10000, 18)


In [65]:
# Question 1: What is the churn rate and average balance grouped by Geography and Gender?
q1_analysis = df_clean.groupby(['geography', 'gender'], observed=False).agg(
    
    total_customers=('customerid', 'count'),
    churned_customers=('exited', 'sum'),
    churn_rate=('exited', 'mean'),
    avg_balance=('balance', 'mean')
).reset_index()
# the code is read to run



# Question 2: How does customer churn rate vary across different Card Types?
q2_analysis = df_clean.groupby('card_type', observed=False).agg(
    
    total_customers=('customerid', 'count'),
    churn_rate=('exited', 'mean'),
    avg_points=('point_earned', 'mean'),
    
    avg_satisfaction=('satisfaction_score', 'mean')
    
).sort_values(by='churn_rate', ascending=False).reset_index()


# Question 3: How does active membership status impact churn across different credit score tiers?

df_clean['credit_tier'] = pd.cut(df_clean['creditscore'], bins=[0, 600, 750, 900], labels=['Low', 'Medium', 'High'])

q3_analysis = df_clean.groupby(['credit_tier', 'isactivemember'], observed=False).agg(
    
    customer_count=('customerid', 'count'),
    churn_rate=('exited', 'mean'),
    avg_balance=('balance', 'mean')
    
).reset_index()

print("== Analysis 1: Geography & Gender ======")
print(q1_analysis)

print("\n===   Analysis 2  : Card Type Impact ===")

print(q2_analysis)

print("\n=== Analysis 3: Credit  Tier & Active Status = ==")

print(q3_analysis)

== Analysis 1: Geography & Gender ======
  geography  gender  total_customers  churned_customers  churn_rate  \
0    France  Female             2261                460    0.203450   
1    France    Male             2753                351    0.127497   
2   Germany  Female             1193                448    0.375524   
3   Germany    Male             1316                366    0.278116   
4     Spain  Female             1089                231    0.212121   
5     Spain    Male             1388                182    0.131124   

     avg_balance  
0   60322.670159  
1   63546.284875  
2  119145.966471  
3  120259.668222  
4   59862.092534  
5   63352.833746  

===   Analysis 2  : Card Type Impact ===
  card_type  total_customers  churn_rate  avg_points  avg_satisfaction
0   DIAMOND             2507    0.217790  605.983646          2.993618
1  PLATINUM             2495    0.203607  608.839679          3.010020
2    SILVER             2496    0.201122  604.002804          3.007212
3 

### Key Analytical Findings & Interpretation:

1. **Geography & Gender Impact:**
   - Customers in **Germany** demonstrate a significantly higher churn rate compared to France and Spain, with female customers in Germany showing the highest churn rate (~37.5%).
   - The average account balance for German customers is also higher, indicating that Germany represents a high-value, high-risk customer segment for the bank.

2. **Card Type Distribution:**
   - Churn rates remain relatively consistent across all credit card tiers (DIAMOND, GOLD, SILVER, PLATINUM) at approximately 20%.
   - Satisfaction scores and points earned show minimal variance across card tiers, suggesting card tier benefits alone are not the primary driver for customer retention.

3. **Active Membership & Credit Tiers:**
   - **Active members (`isactivemember = 1`)** exhibit a dramatically lower churn rate (~14.2%) compared to inactive members (~26.8%) across all credit score tiers.
   - Increasing customer engagement and converting inactive users into active members is the most effective retention strategy regardless of their credit score.

In [66]:
import openpyxl
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
import pandas as pd

# 1. Export cleaned dataset, analysis, and summary into a single Excel file
excel_filename = "Project_Report.xlsx"

with pd.ExcelWriter(excel_filename, engine="openpyxl") as writer:
    # Sheet 1: Cleaned Data
    df_clean.to_excel(writer, sheet_name="Cleaned Data", index=False)

    # Sheet 2: Analysis Results
    q1_analysis.to_excel(writer, sheet_name="Analysis", startrow=0, index=False)
    q2_analysis.to_excel(
        writer,
        sheet_name="Analysis",
        startrow=len(q1_analysis) + 4,
        index=False,
    )
    q3_analysis.to_excel(
        writer,
        sheet_name="Analysis",
        startrow=len(q1_analysis) + len(q2_analysis) + 8,
        index=False,
    )

    # Sheet 3: Executive Summary
    summary_data = pd.DataFrame({
        "Key Metric": [
            "Total Customer Records",
            "Overall Bank Churn Rate",
            "Average Account Balance",
            "Average Credit Score",
            "Active Member Percentage",
        ],
        "Value": [
            len(df_clean),
            f"{df_clean['exited'].mean():.2%}",
            f"${df_clean['balance'].mean():,.2f}",
            f"{df_clean['creditscore'].mean():.1f}",
            f"{df_clean['isactivemember'].mean():.2%}",
        ],
    })
    summary_data.to_excel(writer, sheet_name="Summary", index=False)

# 2. Add metadata header to Summary sheet
wb = openpyxl.load_workbook(excel_filename)
ws_summary = wb["Summary"]

ws_summary.insert_rows(1, 6)

ws_summary["A1"] = "Project Name:"
ws_summary["B1"] = "Bank Customer Churn Analysis & Reporting"

ws_summary["A2"] = "First Student:"
ws_summary["B2"] = "Mustafa Mahmoud Hassouna (ID: 120258131)"

ws_summary["A3"] = "Second Student:"
ws_summary["B3"] = "Hamid Ibrahim Miqdad (ID: 120251119)"

# 3. Define Styling Properties
center_alignment = Alignment(horizontal="center", vertical="center")
header_fill = PatternFill(
    start_color="1F4E78", end_color="1F4E78", fill_type="solid"
)
header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
bold_font = Font(name="Calibri", size=11, bold=True)
regular_font = Font(name="Calibri", size=11)
thin_border = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9"),
)

# Style metadata in Summary sheet
for row in range(1, 4):
    ws_summary[f"A{row}"].font = bold_font
    ws_summary[f"B{row}"].font = regular_font

# 4. Apply formatting across all sheets (Center alignment, Borders, Widths)
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]

    # Format main headers (Row 1 for Cleaned Data & Analysis, Row 7 for Summary)
    header_row_idx = 7 if sheet_name == "Summary" else 1

    for cell in ws[header_row_idx]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = center_alignment

    # Align Data Cells & Set Auto Column Widths
    for col in ws.columns:
        max_len = 0
        col_letter = col[0].column_letter

        for cell in col:
            # Apply alignment and borders to all non-empty cells
            if cell.value is not None:
                cell.alignment = center_alignment
                cell.border = thin_border
                max_len = max(max_len, len(str(cell.value)))

        # Set adjusted width with padding
        ws.column_dimensions[col_letter].width = max(max_len + 4, 12)

# Save formatted workbook
wb.save(excel_filename)
print(
    f"Report successfully exported and styled in '{excel_filename}' with 3 sheets!"
)

Report successfully exported and styled in 'Project_Report.xlsx' with 3 sheets!
